In [1]:
import tensorflow as tf
import keras
import keras_hub
import os
import pandas as pd
from plotnine import ggplot, aes, geom_line, facet_wrap, theme_bw, theme, scale_x_continuous, element_blank, labs
import re
import matplotlib.pyplot as plt
import sklearn as sk
from sklearn.model_selection import train_test_split
import numpy as np

In [4]:
import psutil
ram = psutil.virtual_memory()
print(f"Total: {ram.total / 1e9:.1f} GB")
print(f"Available: {ram.available / 1e9:.1f} GB")
print(f"Used: {ram.used / 1e9:.1f} GB")

Total: 17.0 GB
Available: 9.3 GB
Used: 7.7 GB


In [5]:
columns = ['fitness', 'somaline_genes', 'gravity']

test_df = pd.read_csv(
    r'C:\Users\khatch\Documents\GitHub\data_10_runs_evodevo\merged_master_data.csv',
    usecols=columns,
    skiprows=lambda i: i > 0 and i % 5 == 0 # keeps 80% (skips every 5th row)
)
print(test_df.shape)
print(test_df.memory_usage(deep=True).sum() / 1e9, "GB")

(240000, 3)
4.335600132 GB


In [6]:
test_df = test_df.dropna(subset=['fitness']).reset_index(drop=True)

In [ ]:
# downloading results

# trying out parquet since pandas was struggling
for i, chunk in enumerate(pd.read_csv(
    r'C:\Users\khatch\Documents\GitHub\data_10_runs_evodevo\merged_master_data.csv',
    usecols=columns,
    chunksize=10000
)):
    chunk.to_parquet(f'data_chunk_{i}.parquet')

# load all chunks:
import glob
test_df = pd.concat(
    [pd.read_parquet(f) for f in sorted(glob.glob('data_chunk_*.parquet'))],
    ignore_index=True
)


#columns = ['fitness', 'somaline_genes', 'gravity']

#test_df = pd.read_csv(r'C:\Users\khatch\Documents\GitHub\data_10_runs_evodevo\merged_master_data.csv', 
#                      usecols=columns,
#                     )


In [ ]:
test_df.head()

In [7]:
def plot_fit_history(history):
  # A function that plots model accuracy and loss during training
  df = pd.DataFrame(history.history)
  df['epoch'] = np.arange(len(df))

  # Reshape the dataframe to long format
  df_longer = df.melt(id_vars=['epoch'], var_name="measure", value_name="value")

  # Extract measure name and data type (train or validation)
  df_longer['data_type'] = df_longer['measure'].apply(lambda x: 'validation' if x.startswith('val_') else 'training')
  df_longer['measure'] = df_longer['measure'].apply(lambda x: re.sub('val_', '', x))

  return (ggplot(df_longer, mapping=aes(x="epoch", y="value", color="data_type"))+
   geom_line()+
   scale_x_continuous(breaks=np.arange(0, len(df)))+
   facet_wrap('~measure', scales='free_y')+
   labs(x = "Epoch", y="", color="Dataset")+
   theme_bw()+
   theme(panel_grid=element_blank()))

In [8]:
# all part codes (found in
partMap = {

    "000": "sphere",
    "001": "sphere",
    "002": "size+",
    "003": "size-",

    "010": "sensor+",
    "011": "sensor+",
    "012": "sensor-",
    "013": "joints-",

    "020": "joints+",
    "021": "joints+",
    "022": "neurons+",
    "023": "neurons+",

    "030": "neurons-",
    "031": "inputs+",
    "032": "inputs+",
    "033": "inputs-",

    # ------

    "100": "sensor",
    "101": "output+",
    "102": "output+",
    "103": "output-",

    "110": "sensorX+",
    "111": "sensorX-",
    "112": "sensorY+",
    "113": "sensorY-",

    "120": "sensorZ+",
    "121": "sensorZ-",
    "122": "START",
    "123": "wireConnections+",

    "130": "wireConnections-",
    "131": "mag+60",
    "132": "jointLowerLim+",
    "133": "jointLowerLim-",

    # ------

    "200": "joint",
    "201": "jointMotor+",
    "202": "jointMotor-",
    "203": "jointFree+",

    "210": "sphereJointX+",
    "211": "sphereJointX-",
    "212": "sphereJointY+",
    "213": "sphereJointY-",

    "220": "sphereJointZ+",
    "221": "sphereJointZ-",
    "222": "STOP",
    "223": "mag+90",

    "230": "mag+100",
    "231": "mag+110",
    "232": "mag+50",
    "233": "mag+70",

    # ------


    "300": "jointUpperLimit+",
    "301": "jointUpperLimit-",
    "302": "jointFree-",
    "303": "wire",

    "310": "wireWeight+",
    "311": "wireWeight-",
    "312": "neuron",
    "313": "neuronInputs-",

    "320": "neuronInputs+",
    "321": "neuronInputs+",
    "322": "neuronInputs+",
    "323": "neuronOutputs+",

    "330": "neuronOutputs-",
    "331": "mag+30",
    "332": "mag+40",
    "333": "mag+80",

}

all_codons = sorted(partMap.keys())  # 64 unique codon strings e.g. "000", "001"...
indexMap = {codon: i for i, codon in enumerate(all_codons)}

'''
# I don't know if this is the simplest way, but I'm also making an index dictionary as well
indexMap = {

    "sphere": 0,
    "sphere": 1,
    "size+": 2,
    "size-": 3,

    "sensor+": 4,
    "sensor+": 5,
    "sensor-": 6,
    "joints-": 7,

    "joints+": 8,
    "joints+": 9,
    "neurons+": 10,
    "neurons+": 11,

    "neurons-": 12,
    "inputs+": 13,
    "inputs+": 14,
    "inputs-": 15,

    # ------

    "sensor": 16,
    "output+": 17,
    "output+": 18,
    "output-": 19,

    "sensorX+": 20,
    "sensorX-": 21,
    "sensorY+": 22,
    "sensorY-": 23,

    "sensorZ+": 24,
    "sensorZ-": 25,
    "START": 26,
    "wireConnections+": 27,

    "wireConnections-": 28,
    "mag+60": 29,
    "jointLowerLim+": 30,
    "jointLowerLim-": 31,

    # ------

    "joint": 32,
    "jointMotor+": 33,
    "jointMotor-": 34,
    "jointFree+": 35,

    "sphereJointX+": 36,
    "sphereJointX-": 37,
    "sphereJointY+": 38,
    "sphereJointY-": 39,

    "sphereJointZ+": 40,
    "sphereJointZ-": 41,
    "STOP": 42,
    "mag+90": 43,

    "mag+100": 44,
    "mag+110": 45,
    "mag+50": 46,
    "mag+70": 47,

    # ------


    "jointUpperLimit+": 48,
    "jointUpperLimit-": 49,
    "jointFree-": 50,
    "wire": 51,

    "wireWeight+": 52,
    "wireWeight-": 53,
    "neuron": 54,
    "neuronInputs-": 55,

    "neuronInputs+": 56,
    "neuronInputs+": 57,
    "neuronInputs+": 58,
    "neuronOutputs+": 59,

    "neuronOutputs-": 60,
    "mag+30": 61,
    "mag+40": 62,
    "mag+80": 63,

}
'''

'\n# I don\'t know if this is the simplest way, but I\'m also making an index dictionary as well\nindexMap = {\n\n    "sphere": 0,\n    "sphere": 1,\n    "size+": 2,\n    "size-": 3,\n\n    "sensor+": 4,\n    "sensor+": 5,\n    "sensor-": 6,\n    "joints-": 7,\n\n    "joints+": 8,\n    "joints+": 9,\n    "neurons+": 10,\n    "neurons+": 11,\n\n    "neurons-": 12,\n    "inputs+": 13,\n    "inputs+": 14,\n    "inputs-": 15,\n\n    # ------\n\n    "sensor": 16,\n    "output+": 17,\n    "output+": 18,\n    "output-": 19,\n\n    "sensorX+": 20,\n    "sensorX-": 21,\n    "sensorY+": 22,\n    "sensorY-": 23,\n\n    "sensorZ+": 24,\n    "sensorZ-": 25,\n    "START": 26,\n    "wireConnections+": 27,\n\n    "wireConnections-": 28,\n    "mag+60": 29,\n    "jointLowerLim+": 30,\n    "jointLowerLim-": 31,\n\n    # ------\n\n    "joint": 32,\n    "jointMotor+": 33,\n    "jointMotor-": 34,\n    "jointFree+": 35,\n\n    "sphereJointX+": 36,\n    "sphereJointX-": 37,\n    "sphereJointY+": 38,\n    "sph

In [9]:
# defining part types and setting up dictionaries for them as well
part_types = {'sphere', 'sensor', 'joint', 'neuron', 'wire'}
part_type_index = {
    'sphere':  0,
    'sensor':  1,
    'joint':   2,
    'neuron':  3,
    'wire':    4
}

In [10]:
gravity_encoding = {
    '-4':  0,
    '-7':  1,
    '-10': 2,
    '-13': 3,
    '-16': 4
}

In [11]:
maxGenes = 55

# feature size determiined by number of codons + number of parts
# + gene length value + has_part bool value + grav value
geneVectorDimension = len(partMap) + len(part_type_index) + 1 + 1

In [12]:
def build_model_arrays(dataframe):
    # converts the pd df into model data

    n_individuals = len(dataframe)

    # create all the arrays
    X_gene_sequences = np.zeros(
        (n_individuals, maxGenes, geneVectorDimension),
        dtype=np.float32
    )
    X_padding_masks = np.zeros(
        (n_individuals, maxGenes),
        dtype=bool
    )
    X_gravity = np.zeros(
        (n_individuals, 1),
        dtype=np.float32
    )
    y_fitness = np.zeros(
        n_individuals,
        dtype=np.float32
    )

    # Iterate over each individual (each row in the dataframe)
    for row_index, row in dataframe.iterrows():

        # --- Parse genome string into codon integers ---
        genome_string = row['somaline_genes']
        codon_list = feature_encodings(genome_string)

        # --- Convert codon list into padded gene sequence + mask ---
        # genome_to_sequence() is the function defined in the embedding code
        gene_sequence, padding_mask = genome_to_sequence(genome_string)

        # --- Store in pre-allocated arrays ---
        # row_index from iterrows() may not be 0-based if the dataframe
        # was filtered or sliced, so we use enumerate via a separate counter
        X_gene_sequences[row_index] = gene_sequence
        X_padding_masks[row_index]  = padding_mask
        X_gravity[row_index, 0]     = row['gravity']
        y_fitness[row_index]        = row['fitness']

    return X_gene_sequences, X_padding_masks, X_gravity, y_fitness

In [13]:
def parse_somaline_genes(gene_string):
  # returns counts for bodies, sensors, neurons, and wires based on somaline genes
  # essentially a helper function for feature_encodings

  # creating the return list
  expressed_parts_list = []

  # start and stop codons
  start = "122"
  stop = "222"

  # convert the 18,000 digits into 6,000 triplet codons
  codons = [gene_string[i:i+3] for i in range(0, len(gene_string), 3)]

  # setting up necessary variables for the following loop
  genes = []
  current_gene = []
  in_gene = False
  part_expressed_in_current_gene = False

  for codon in codons:
    # start codon
    if codon == start:
      in_gene = True
      current_gene = []

    # stop codon
    elif codon == stop and in_gene:
      if len(current_gene) > 0:
          genes.append(current_gene)
      in_gene = False
      current_gene = []

    # everything in between start and stop
    elif in_gene:
      current_gene.append(codon)

  return genes

In [14]:
def feature_encodings(gene_codons):
    # a fcuntion that converts a list of codons from one gene into a feature vector
    # for the neural net to train on

    # indicates counts of all 64 codons, gene length (number of codons),
    # part presence, part type, and gravity condityion


    category_counts = np.zeros(64, dtype=np.float32)
    part_type_counts = np.zeros(5, dtype=np.float32)
    gene_length = len(gene_codons)
    has_part = 0.0

    for codon in gene_codons:
      # increment category count by 1
      codon_index = indexMap.get(codon, -1)
      if codon_index >= 0:
          category_counts[codon_index] += 1
          
      category = partMap.get(codon, 'unknown')
      if category in part_types:
        # notes if a gene has a part
        part_index = part_type_index[category]
        part_type_counts[part_index] = 1.0
        has_part = 1.0

#--
    # normalize counts to compare on the same scale
    if gene_length > 0:
        category_counts = category_counts / gene_length
    normalized_gene_length = gene_length / maxGenes

    # concatenate into one flat feature vector.
    gene_feature_vector = np.concatenate([
        category_counts,
        part_type_counts,
        [normalized_gene_length],
        [has_part]
    ])  # final shape: (64 + 5 + 2,)

    return gene_feature_vector

In [15]:
def genome_to_sequence(genome_string):
    list_of_genes = parse_somaline_genes(genome_string)   # takes raw string

    gene_sequence = np.zeros(
        (maxGenes, geneVectorDimension),
        dtype=np.float32
    )

    totalGenes = min(len(list_of_genes), maxGenes)
    padding_mask = np.zeros(maxGenes, dtype=bool)

    for gene_index in range(totalGenes):
        this_gene_codons = list_of_genes[gene_index]
        gene_sequence[gene_index] = feature_encodings(this_gene_codons)  # no grav
        padding_mask[gene_index] = True

    return gene_sequence, padding_mask

In [16]:
D_MODEL    = 64    # size of the transformer's internal hidden dimension
NUM_HEADS  = 4     # number of parallel attention heads
NUM_LAYERS = 2     # number of stacked transformer blocks
FF_DIM     = 128   # size of the feed-forward layer inside each transformer block

# Defined earlier:
# maxGenes = 55
# geneVectorDimension = 72

# =============================================================================
# POSITIONAL ENCODING
# =============================================================================
# Transformers have no built-in sense of order. Positional encoding adds a
# signal to each gene position so the model knows where in the genome each
# gene sits. This is computed once and added to the projected gene embeddings.
# Output shape: (1, MAX_GENES, D_MODEL) — the leading 1 allows broadcasting
# across the batch dimension during training.

position_indices = np.arange(maxGenes)[:, np.newaxis]        # shape: (MAX_GENES, 1)
dimension_indices = np.arange(D_MODEL)[np.newaxis, :]         # shape: (1, D_MODEL)
angle_rates = position_indices / np.power(
    10000,
    (2 * (dimension_indices // 2)) / D_MODEL
)
angle_rates[:, 0::2] = np.sin(angle_rates[:, 0::2])           # sin applied to even dims
angle_rates[:, 1::2] = np.cos(angle_rates[:, 1::2])           # cos applied to odd dims
positional_encoding = angle_rates[np.newaxis, :, :]            # shape: (1, MAX_GENES, D_MODEL)
positional_encoding = positional_encoding.astype(np.float32)



# THE MODEL -------------------------------

# --- Inputs ---
gene_input = keras.layers.Input(shape=(maxGenes, geneVectorDimension))

# padding_mask_input: boolean mask marking which positions are real genes
# vs zero-padded positions (true = real gene, false = padding)
padding_mask_input = keras.layers.Input(shape=(maxGenes,),dtype='bool',name='padding_mask')

# adding in gravity variable here
gravity_input = keras.layers.Input(shape=(1,))

# -----

# turns all inputs into a dimension the transformer can handle +. adds positional encoding from above
dense1 = keras.layers.Dense(D_MODEL)(gene_input)
dense1 = dense1 + positional_encoding

# dropout after embedding
dropout1 = keras.layers.Dropout(0.1)(dense1)

# transformer 1
padding_mask_2d = keras.layers.Lambda(
    lambda m: keras.ops.cast(m, 'float32')[:, None, :] *
              keras.ops.cast(m, 'float32')[:, :, None],
    name='attn_mask_2d'
)(padding_mask_input)

attn_output_1 = keras.layers.MultiHeadAttention(num_heads=NUM_HEADS, key_dim=(D_MODEL // NUM_HEADS), dropout=0.1)(dropout1, dropout1, attention_mask=padding_mask_2d)

# normalize to reduce vanishing gradients
norm1 = keras.layers.LayerNormalization(epsilon=1e-6)(dropout1 + attn_output_1)

# Feed-forward block: two Dense layers applied position-wise (independently
# to each gene position). Expands to FF_DIM then projects back to D_MODEL.
dense21 = keras.layers.Dense(FF_DIM, activation='gelu')(norm1)
dense22 = keras.layers.Dense(D_MODEL, name='ff1_project')(dense21)
dropout21 = keras.layers.Dropout(0.1, name='ff1_dropout')(dense22)

# Second residual connection + normalization after the feed-forward block.
norm2 = keras.layers.LayerNormalization(epsilon=1e-6)(norm1 + dropout21)

# -----

# transformer 2
attn_output_2 = keras.layers.MultiHeadAttention(num_heads=NUM_HEADS,key_dim=(D_MODEL // NUM_HEADS), dropout=0.1)(norm2, norm2, attention_mask=padding_mask_2d)
norm3 = keras.layers.LayerNormalization(epsilon=1e-6)(norm2 + attn_output_2)

dense31 = keras.layers.Dense(FF_DIM, activation='gelu')(norm3)
dense32 = keras.layers.Dense(D_MODEL)(dense31)
dropout31 = keras.layers.Dropout(0.1)(dense32)

norm4 = keras.layers.LayerNormalization(epsilon=1e-6)(norm3 + dropout31)

# -----

# Calculates metrics while dividing by number of real genes (excludes masked genes)
mask_float = keras.layers.Lambda(
    lambda m: keras.ops.cast(m, 'float32')[:, :, None],
    name='mask_cast'
)(padding_mask_input)

masked_sum = keras.layers.Lambda(
    lambda inputs: keras.ops.sum(inputs[0] * inputs[1], axis=1),
    name='masked_sum'
)([norm4, mask_float])

real_gene_count = keras.layers.Lambda(
    lambda m: keras.ops.sum(keras.ops.cast(m, 'float32')[:, :, None], axis=1),
    name='gene_count'
)(padding_mask_input)

pooled_genome = keras.layers.Lambda(
    lambda inputs: inputs[0] / keras.ops.maximum(inputs[1], 1e-7),
    name='masked_mean'
)([masked_sum, real_gene_count])

# -----

# concatnate gravity and the pooled genome
genome_and_gravity = keras.layers.Concatenate()([pooled_genome, gravity_input])

# Standard dense layers to for prediction (this is now all outside of the attention model)
dense0 = keras.layers.Dense(64, activation='relu', kernel_regularizer=keras.regularizers.L2())(genome_and_gravity)
dropout0 = keras.layers.Dropout(0.2)(dense0)

dense01 = keras.layers.Dense(32, activation='relu', kernel_regularizer=keras.regularizers.L2())(dropout0)
dropout01 = keras.layers.Dropout(0.2, name='head_dropout2')(dense01)

# Single output neuron — predicting fitness as a scalar continuous value
output = keras.layers.Dense(1)(dropout01)

# --------

model = keras.Model(
    inputs=[gene_input, padding_mask_input, gravity_input],
    outputs=output,
    name='genome_transformer'
)

model.summary()

Model: "genome_transformer"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 55, 71)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 55, 64)    │      4,608 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 55, 64)    │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ padding_mask        │ (None, 55)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 55, 64)    │          0 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attn_mask_2d        │ (None, 55, 55)    │          0 │ padding_mask[0][… │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 55, 64)    │     16,640 │ dropout[0][0],    │
│ (MultiHeadAttentio… │                   │            │ dropout[0][0],    │
│                     │                   │            │ attn_mask_2d[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 55, 64)    │          0 │ dropout[0][0],    │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 55, 64)    │        128 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 55, 128)   │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ff1_project (Dense) │ (None, 55, 64)    │      8,256 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ff1_dropout         │ (None, 55, 64)    │          0 │ ff1_project[0][0] │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 55, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ ff1_dropout[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 55, 64)    │        128 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 55, 64)    │     16,640 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
│                     │                   │            │ attn_mask_2d[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 55, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 55, 64)    │        128 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 55, 128)   │      8,320 │ layer_normalizat

 Total params: 77,889 (304.25 KB)

 Trainable params: 77,889 (304.25 KB)

 Non-trainable params: 0 (0.00 B)

In [17]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4, clipnorm=1.0),
    loss='mse',
    metrics=['mae']
)

checkpoint_callback = keras.callbacks.ModelCheckpoint(
    filepath='NORMALIZED_genome_transformer_best.keras',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

# early stopping in case of overfitting
early_stopping_callback = keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)

In [18]:
# pre-computing genes to speed up training time (I want to stop the transformer from 
# calculating the gene sequence while training)
sequence_cache = {}
for genome_string in test_df['somaline_genes'].unique():
    sequence_cache[genome_string] = genome_to_sequence(genome_string)

In [19]:
import keras
import numpy as np

class GenomeDataGenerator(keras.utils.Sequence):
    """
    Generates batches of (gene_sequence, padding_mask, gravity) → fitness
    on the fly from the raw dataframe. Only one batch lives in RAM at a time.
    """

    def __init__(self, dataframe, batch_size=32, shuffle=True,  **kwargs):
        """
        Parameters
        ----------
        dataframe  : pd.DataFrame with columns somaline_genes, fitness, gravity
        batch_size : number of individuals per batch
        shuffle    : whether to shuffle the order of individuals each epoch
        """
        super().__init__(**kwargs)   # ← add this
        self.dataframe  = dataframe.reset_index(drop=True)
        self.batch_size = batch_size
        self.shuffle    = shuffle
        self.indices    = np.arange(len(self.dataframe))
        self.on_epoch_end()

    def __len__(self):
        """Number of batches per epoch."""
        return int(np.ceil(len(self.dataframe) / self.batch_size))        

    def __getitem__(self, batch_number):
        """
        Generates one batch of data.

        Parameters
        ----------
        batch_number : int, the index of the batch to generate

        Returns
        -------
        inputs : list of three arrays
            [gene_sequences, padding_masks, gravity_values]
            shapes: (batch_size, MAX_GENES, GENE_FEATURE_DIM)
                    (batch_size, MAX_GENES)
                    (batch_size, 1)

        targets : np.ndarray of shape (batch_size,)
            Fitness values for this batch
        """
        # Determine which row indices belong to this batch
        batch_start = batch_number * self.batch_size
        batch_end   = min(batch_start + self.batch_size, len(self.dataframe))
        batch_indices = self.indices[batch_start : batch_end]
        actual_batch_size = len(batch_indices)

        # Pre-allocate batch arrays — only batch_size rows, not 300,000
        batch_gene_sequences = np.zeros(
            (actual_batch_size, maxGenes, geneVectorDimension),
            dtype=np.float32
        )
        batch_padding_masks = np.zeros(
            (actual_batch_size, maxGenes),
            dtype=bool
        )
        batch_gravity = np.zeros(
            (actual_batch_size, 1),
            dtype=np.float32
        )
        batch_fitness = np.zeros(
            actual_batch_size,
            dtype=np.float32
        )

        # Fill the batch arrays one individual at a time
        for position_in_batch, row_index in enumerate(batch_indices):
            row = self.dataframe.iloc[row_index]

            codon_list = feature_encodings(row['somaline_genes'])
            gene_sequence, padding_mask = sequence_cache[row['somaline_genes']]

            batch_gene_sequences[position_in_batch] = gene_sequence
            batch_padding_masks[position_in_batch]  = padding_mask
            batch_gravity[position_in_batch, 0]     = row['gravity']
            batch_fitness[position_in_batch]         = row['fitness']

        inputs  = (batch_gene_sequences, batch_padding_masks, batch_gravity)
        targets = batch_fitness
        return inputs, targets

    def on_epoch_end(self):
        """Shuffle the individual ordering at the end of each epoch."""
        if self.shuffle:
            np.random.shuffle(self.indices)


# =============================================================================
# CREATE TRAIN AND TEST GENERATORS
# =============================================================================

# capping the test set and normalizing to remove fitness outliers (currently the max is 
# SIGNIFICANTLY higher than any of the average values)

# cutting off at 99th percentile
upper_cap = test_df['fitness'].quantile(0.99) 
test_df['fitness'] = test_df['fitness'].clip(upper=upper_cap)
# normalizing
fitness_mean = test_df['fitness'].mean()
fitness_std  = test_df['fitness'].std()
test_df['fitness'] = (test_df['fitness'] - fitness_mean) / fitness_std


from sklearn.model_selection import train_test_split

# Split the dataframe itself — not the arrays
train_df, test_df_split = train_test_split(
    test_df,
    test_size=0.2,
    random_state=42
)

print(test_df['fitness'].isna().sum())
print(test_df['somaline_genes'].isna().sum())
print(test_df['gravity'].isna().sum())
print(np.isinf(test_df['fitness']).sum())

train_generator = GenomeDataGenerator(train_df, batch_size=32, shuffle=True)
test_generator  = GenomeDataGenerator(test_df_split, batch_size=32, shuffle=False)

0
0
0
0


In [21]:
train_history = model.fit(
    train_generator,
    validation_data=test_generator,
    epochs=100,
    callbacks=[early_stopping_callback, checkpoint_callback]
)

Epoch 1/100
5999/6000 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step - loss: 1.5831 - mae: 0.8110
Epoch 1: val_loss improved from None to 1.06776, saving model to NORMALIZED_genome_transformer_best.keras

Epoch 1: finished saving model to NORMALIZED_genome_transformer_best.keras
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 817s 135ms/step - loss: 1.3028 - mae: 0.7955 - val_loss: 1.0678 - val_mae: 0.7918
Epoch 2/100
5999/6000 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step - loss: 1.0334 - mae: 0.7910
Epoch 2: val_loss improved from 1.06776 to 0.99802, saving model to NORMALIZED_genome_transformer_best.keras

Epoch 2: finished saving model to NORMALIZED_genome_transformer_best.keras
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 811s 135ms/step - loss: 1.0141 - mae: 0.7901 - val_loss: 0.9980 - val_mae: 0.7922
Epoch 3/100
5999/6000 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step - loss: 0.9765 - mae: 0.7866
Epoch 3: val_loss improved from 0.99802 to 0.97912, saving model to NORMALIZED_genome_transformer_best.keras

Epoch 3: finished saving model to NORMA

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



5695/6000 ━━━━━━━━━━━━━━━━━━━━ 32s 107ms/step - loss: 0.9463 - mae: 0.7807

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



3242/6000 ━━━━━━━━━━━━━━━━━━━━ 4:56 108ms/step - loss: 0.9432 - mae: 0.7801

...

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



5598/6000 ━━━━━━━━━━━━━━━━━━━━ 43s 109ms/step - loss: 0.9188 - mae: 0.7670

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



6000/6000 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step - loss: 0.9152 - mae: 0.7648
Epoch 40: val_loss did not improve from 0.93420
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 862s 144ms/step - loss: 0.9179 - mae: 0.7665 - val_loss: 0.9363 - val_mae: 0.7820
Epoch 41/100
 580/6000 ━━━━━━━━━━━━━━━━━━━━ 10:30 116ms/step - loss: 0.9246 - mae: 0.7705

KeyboardInterrupt: 

In [20]:
print(test_df['fitness'].describe())

# end conditions (epoch 40 end: loss: 0.9179 - mae: 0.7665 - val_loss: 0.9363 - val_mae: 0.7820)

count    2.399900e+05
mean    -1.459042e-16
std      1.000000e+00
min     -1.133944e+00
25%     -8.166578e-01
50%     -2.470518e-01
75%      5.639290e-01
max      3.167716e+00
Name: fitness, dtype: float64


In [ ]:
'''
# Creating the Model

# Basic neural network
input = keras.layers.Input(shape = (6,))
dense1 = keras.layers.Dense(250, activation = "relu", kernel_regularizer = keras.regularizers.L2())(input)
dropout1 = keras.layers.Dropout(0.2)(dense1)
dense2 = keras.layers.Dense(250, activation = "relu", kernel_regularizer = keras.regularizers.L2())(dropout1)
dropout2 = keras.layers.Dropout(0.2)(dense2)
dense3 = keras.layers.Dense(250, activation = "relu", kernel_regularizer = keras.regularizers.L2())(dropout2)
output = keras.layers.Dense(1)(dense3)

model = keras.Model(inputs=input, outputs=output)
model.summary()

# -----

# compiling the model
model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

# early stopping in case of overfitting
#early_stopping_callback = keras.callbacks.EarlyStopping(monitor="val_loss", patience = 4)

# ----

train_history = model.fit(x = X_train, y = y_train,
                            validation_data=(X_test, y_test),
                            epochs = 100,
                            batch_size = 32,
                            #callbacks=[early_stopping_callback]
                            )
'''

In [23]:
plot_fit_history(train_history)

NameError: name 'train_history' is not defined

In [ ]:
test_df['predictions'] = model.predict(test_df[[
    'gravity',
    'Sensor',
    'Body (Sphere)',
    'Joint',
    'Neuron',
    'Wire']])

In [ ]:
test_df

In [ ]:
test_df['mse'] = (((test_df['predictions']) - (test_df['fitness'])) ** 2) ** (0.5)

print("mean error: " + str(test_df['mse'].mean()))